In [ ]:
import os
import glob

# 1) Disable XNNPACK BEFORE importing TF (debug only)
os.environ["TF_TFLITE_USE_XNNPACK"] = "0"

import numpy as np
import matplotlib.pyplot as plt
import cv2
from tflite_inference import TFLiteInference

%load_ext autoreload
%autoreload 2

In [ ]:
MODEL_FP32_PTH = "/home/klymy/projects/researches/people_detection_pg2/results/cam3_revB_batch1_5_margin_15_rq_bg_wb_alif_clahe_box_01_dfl_01_2nd/weights/best_non_separated_saved_model_full_npu/best_float32.tflite"
MODEL_UINT8_PTH = "/home/klymy/projects/researches/people_detection_pg2/results/cam3_revB_batch1_5_margin_15_rq_bg_wb_alif_clahe_box_01_dfl_01_2nd/weights/best_non_separated_saved_model_full_npu/best_full_integer_quant.tflite"

BASE_PTH = "/home/klymy/data/people_detection/filtered_v3_margin_15/images/val/"
IMGSZ = 320

In [ ]:
# Initialize inference class with layer indices to collect
# You can specify additional layer indices to collect during inference
layer_indices = []  # Add layer indices here if needed, e.g., [100, 200, 300]

fp32_inference = TFLiteInference(MODEL_FP32_PTH, is_uint8=False, imgsz=IMGSZ, layer_indices=layer_indices)
uint8_inference = TFLiteInference(MODEL_UINT8_PTH, is_uint8=True, imgsz=IMGSZ, layer_indices=layer_indices)

In [ ]:
# Load and run inference on first image
img_pths = glob.glob(os.path.join(BASE_PTH, "*.jpg"))
print(f"Found {len(img_pths)} images")

img = cv2.imread(img_pths[0])
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

In [ ]:
fp32_output = fp32_inference.infer_image(img.astype(np.float32) / 255)
uint8_output = uint8_inference.infer_image(img.astype(np.uint8))

In [ ]:
fp32_logits_np, fp32_probs_np = fp32_inference.get_logits_probs()
uint8_logits_np, uint8_probs_np = uint8_inference.get_logits_probs()

In [ ]:
idx = 684

idxs = np.arange(len(uint8_logits_np[0]))
np.random.seed(42)
np.random.shuffle(idxs)

for i in idxs[:10]:
    plt.figure(figsize=(15, 4))
    plt.subplot(1, 2, 1)
    plt.plot(uint8_logits_np[0, i], 'ro-', label='uint8 model')
    plt.plot(fp32_logits_np[0, i], 'bo-', label='float32 model')
    plt.legend()
    plt.title(f'Logits from TFLite models: {i}')

    plt.subplot(1, 2, 2)
    plt.plot(uint8_probs_np[0, i], 'ro-', label='uint8 model')
    plt.plot(fp32_probs_np[0, i], 'bo-', label='float32 model')
    plt.legend()
    plt.title('Probabilities from TFLite models')

In [ ]:
uint8_scores = uint8_output["outputs"][0]
fp32_scores = fp32_output["outputs"][1]

uint8_coordinates = uint8_output["outputs"][1]
fp32_coordinates = fp32_output["outputs"][0]

idxs = np.where(uint8_scores[0] > 0.5)[0]
len(idxs)

In [ ]:
plt.figure(figsize=(16, 4))
plt.plot(uint8_scores.flatten(), 'r.', label='uint8 model')
plt.plot(fp32_scores.flatten(), 'b.', label='float32 model', alpha=0.5)
plt.legend()
plt.title('Scores from TFLite models')

In [ ]:
for i in idxs:
    tmp_uint8_box = uint8_coordinates[0, i]
    tmp_fp32_box = fp32_coordinates[0, i]
    
    tmp_uint8_score = uint8_scores[0, i][0]
    tmp_fp32_score = fp32_scores[0, i][0]
    
    plt.figure(figsize=(6, 4))

    plt.plot(tmp_uint8_box, 'ro-', label=f'uint8 score: {tmp_uint8_score:.3f}')
    plt.plot(tmp_fp32_box, 'bo-', label=f'fp32 score: {tmp_fp32_score:.3f}')
    plt.legend()
    plt.title(f'Box coordinates for detection index {i}')
    